# YOLOX Polygon Training: Thermal Cheetah Dataset

This notebook demonstrates how to train YOLOX with **polygon bounding box** support using the Thermal Cheetah dataset.

### Interactive Features
- **Real-time Loss Plots**: Training and Validation loss curves update every epoch.
- **Live Visualization**: See polygon predictions on validation images after each epoch.
- **Interactive Progress**: `tqdm` progress bars and live logs for better visibility.

In [ ]:
import os
import sys
from pathlib import Path
import torch
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output, display, update_display
import cv2
from tqdm.notebook import tqdm
from loguru import logger

# 1. Setup paths
project_root = str(Path(os.getcwd()).absolute())
if project_root not in sys.path:
    sys.path.append(project_root)
os.environ['PYTHONPATH'] = f"{project_root}{os.pathsep}{os.environ.get('PYTHONPATH', '')}"

from yolox.exp import get_exp
from yolox.core import Trainer
from yolox.utils import setup_logger, vis
from yolox.utils.boxes import postprocess, postprocess_polygon
from yolox.data.data_augment import ValTransform

print(f"Project root: {project_root}")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Capture original stdout/err for notebook logging
notebook_stdout = sys.stdout
notebook_stderr = sys.stderr

In [ ]:
def vis_poly(img, bboxes, scores, cls_ids, conf=0.5, class_names=None, color=(0, 255, 0)):
    """Visualize polygon predictions."""
    for i in range(len(bboxes)):
        score = scores[i] if scores is not None else 1.0
        if score < conf:
            continue
        
        poly = np.array(bboxes[i], dtype=np.int32).reshape((-1, 2))
        cls_id = int(cls_ids[i])
        
        cv2.polylines(img, [poly], isClosed=True, color=color, thickness=2)
        
        if class_names is not None:
            if 0 <= cls_id < len(class_names):
                cls_name = class_names[cls_id]
            else:
                cls_name = f"ID:{cls_id}"
        else:
            cls_name = f"{cls_id}"
        
        text = f"{cls_name}: {score:.2f}"
        x, y = poly[0]
        if scores is not None:
            cv2.putText(img, text, (int(x), int(y) - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)
    return img

class InteractiveTrainer(Trainer):
    def __init__(self, exp, args):
        super().__init__(exp, args)
        self.train_losses = []
        self.val_losses = []
        self.maps = []
        self.epochs = []
        
        # Rescue logging for notebook
        logger.remove()
        logger.add(notebook_stderr, format="<green>{time:HH:mm:ss}</green> | <level>{message}</level>", level="INFO")
        logger.info("Logger optimized for notebook.")

    def train_in_iter(self):
        # Use tqdm for iterations
        pbar = tqdm(range(self.max_iter), desc=f"Epoch {self.epoch+1}/{self.max_epoch}", leave=False)
        for self.iter in pbar:
            self.before_iter()
            self.train_one_iter()
            self.after_iter()
            
            # Update pbar with latest loss, handle None during first iter
            if "total_loss" in self.meter.keys():
                latest_loss = self.meter['total_loss'].latest
                if latest_loss is not None:
                    pbar.set_postfix({"loss": f"{latest_loss:.2f}"})
                else:
                    pbar.set_postfix({"loss": "waiting..."})

    def after_epoch(self):
        super().after_epoch()
        
        # Record training loss
        total_loss = self.meter["total_loss"].avg if "total_loss" in self.meter.keys() else 0
        self.train_losses.append(total_loss)
        self.epochs.append(self.epoch + 1)
        
        # Calculate Validation Loss explicitly
        val_loss = self.get_val_loss()
        self.val_losses.append(val_loss)
        
        # Get mAP
        ap50_95, ap50, summary = self.exp.eval(self.model, self.evaluator, self.is_distributed)
        self.maps.append(ap50)
        
        # Update UI
        self.update_notebook_ui()

    @torch.no_grad()
    def get_val_loss(self):
        self.model.train()
        total_val_loss = 0
        num_batches = 0
        val_loader = self.evaluator.dataloader
        
        for imgs, targets, _, _ in val_loader:
            imgs = imgs.to(self.device).to(self.data_type)
            targets = targets.to(self.device).to(self.data_type)
            outputs = self.model(imgs, targets)
            total_val_loss += outputs["total_loss"].item()
            num_batches += 1
            if num_batches >= 5: break
                
        self.model.eval()
        return total_val_loss / num_batches if num_batches > 0 else 0

    def update_notebook_ui(self):
        # Don't clear EVERYTHING, just the UI part if possible
        # But for simplicity, we clear and log results
        print(f"\n[STATUS] Epoch {self.epoch+1} Completed. Train Loss: {self.train_losses[-1]:.4f}, Val Loss: {self.val_losses[-1]:.4f}, mAP@50: {self.maps[-1]:.4f}")
        
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
        
        # Plot Loss
        ax1.plot(self.epochs, self.train_losses, label='Train Loss', marker='o')
        ax1.plot(self.epochs, self.val_losses, label='Val Loss', marker='x', linestyle='--')
        ax1.set_title('Loss Curves')
        ax1.set_xlabel('Epoch')
        ax1.set_ylabel('Loss')
        ax1.grid(True); ax1.legend()
        
        # Plot mAP
        ax2.plot(self.epochs, self.maps, label='mAP@50', color='green', marker='s')
        ax2.set_title('Validation mAP@50')
        ax2.set_xlabel('Epoch')
        ax2.set_ylabel('mAP')
        ax2.grid(True); ax2.legend()
        
        plt.show()
        self.visualize_predictions()

    @torch.no_grad()
    def visualize_predictions(self, num_images=2):
        self.model.eval()
        val_loader = self.evaluator.dataloader
        dataset = val_loader.dataset
        
        for imgs, _, info_imgs, ids in val_loader:
            imgs = imgs.to(self.device).to(self.data_type)
            outputs = self.model(imgs)
            
            if self.exp.use_polygon:
                outputs = postprocess_polygon(outputs, self.exp.num_classes, self.exp.test_conf, self.exp.nmsthre)
            else:
                outputs = postprocess(outputs, self.exp.num_classes, self.exp.test_conf, self.exp.nmsthre)
            
            fig, axes = plt.subplots(1, min(num_images, len(imgs)), figsize=(15, 7))
            if num_images == 1: axes = [axes]
                
            for i in range(min(num_images, len(imgs))):
                img = imgs[i].cpu().numpy().transpose(1, 2, 0)
                img = np.ascontiguousarray(img, dtype=np.uint8)
                
                # --- Plot Ground Truth (Blue) ---
                try:
                    img_id = int(ids[i])
                    ann_ids = dataset.coco.getAnnIds(imgIds=[img_id])
                    anns = dataset.coco.loadAnns(ann_ids)
                    
                    gt_bboxes = []
                    gt_cls_ids = []
                    
                    origin_shape = info_imgs[i]
                    input_size = self.exp.test_size
                    r = min(input_size[0] / origin_shape[0], input_size[1] / origin_shape[1])
                    
                    for ann in anns:
                        if 'segmentation' in ann:
                            seg = ann['segmentation']
                            for poly_points in seg:
                                try:
                                    poly = np.array(poly_points).reshape(-1, 2)
                                    poly = poly * r
                                    gt_bboxes.append(poly.flatten())
                                    gt_cls_ids.append(ann['category_id'])
                                except Exception:
                                    pass
                    
                    if gt_bboxes:
                        img = vis_poly(img, gt_bboxes, None, gt_cls_ids, conf=0.0, class_names=self.exp.class_names, color=(255, 0, 0)) # Blue
                except Exception as e:
                    pass

                # --- Plot Predictions (Green) ---
                if outputs[i] is not None:
                    output = outputs[i].cpu().numpy()
                    bboxes = output[:, 0:8] if self.exp.use_polygon else output[:, 0:4]
                    scores = output[:, 4] * output[:, 5]
                    cls_ids = output[:, 6]
                    
                    if self.exp.use_polygon:
                        img = vis_poly(img, bboxes, scores, cls_ids, self.exp.test_conf, self.exp.class_names, color=(0, 255, 0)) # Green
                    else:
                        img = vis(img, bboxes, scores, cls_ids, self.exp.test_conf, self.exp.class_names)
                
                axes[i].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
                axes[i].set_title(f"Epoch {self.epoch+1} - Val Sample")
                axes[i].axis('off')
            
            plt.tight_layout(); plt.show()
            break
        self.model.train()

## 3. Run Training Loop

In [ ]:
class Args:
    def __init__(self):
        self.batch_size = 2 if device == "cpu" else 4
        self.devices = None
        self.experiment_name = "yolox_notebook_run"
        self.fp16 = False
        self.cache = None
        self.occupy = False
        self.logger = "tensorboard"
        self.ckpt = None
        self.resume = False
        self.start_epoch = None
        self.num_machines = 1
        self.machine_rank = 0
        self.dist_backend = "nccl"
        self.dist_url = None

exp = get_exp("exps/example/yolox_thermal_cheetah_poly.py")
exp.max_epoch = 10 
exp.print_interval = 1
exp.eval_interval = 1
exp.class_names = ["cheetah", "person", "other"]

args = Args()
trainer = InteractiveTrainer(exp, args)
trainer.train()